## Import Libraries

In [32]:
#Import Standard Libraries
import pandas as pd
import numpy as np
from pydub import AudioSegment

In [39]:
#GTZAN expects files in the format of `gtzan_genre`.`5 digit number`.wav
#Need to make customized gtzan or just use custom 5 digit numbers
#Aim for format of `gtzan_genre`.`5 digit number`.`part number`.wav 
def split_audio_overlap_v1(file_path, segment_ms, overlap_ms):
    segments = []
    try:
        audio = AudioSegment.from_file(file_path)
        _, file_destination = file_path.split("/", 1)
        final_file_path, _ = file_destination.split(".wav")
    
        new_file_path = f"gtzan_{segment_ms}ms/{final_file_path}"
    
        # Calculate the step size (segment length - overlap)
        step = segment_ms - overlap_ms
    
        for start in range(0, len(audio), step):
            end = start + segment_ms
            segment = audio[start:end]
            segments.append(segment)
            file_chunk_name = f"{new_file_path}.{start//step}.wav"
            # Optional: export each segment
            segment.export(file_chunk_name, format="wav")
    except Exception as e:
        print(e)
        print(f"{file_path} failed to load")

    return segments

#Treating each audio segment as its own file and giving it its own 5 digit code
#For testing 30ms with 10ms overlap, will randomize the files and select the first 25,000 segments (250k total for dataset) 
file_num = 0

def split_audio_overlap(file_path, segment_ms, overlap_ms, file_num):
    segments = []
    try:
        audio = AudioSegment.from_file(file_path)
        file_path = file_path.removeprefix("gtzan/data/genres/")
        file_path = file_path.removesuffix(".wav")
        #genre/filename -> #genre/genre.num.wav
        
        genre, filename = file_path.split("/")
    
        # Calculate the step size (segment length - overlap)
        step = segment_ms - overlap_ms
    
        for start in range(0, len(audio), step):
            end = start + segment_ms
            segment = audio[start:end]
            segments.append(segment)
            
            new_file_path = f"gtzan_{segment_ms}ms_{overlap_ms}ms_overlap/data/genres/{genre}/{genre}.{file_num:05d}"
            file_num += 1
            
            file_chunk_name = f"{new_file_path}.wav"
            #print(file_chunk_name)
            # Optional: export each segment
            segment.export(file_chunk_name, format="wav")
            """
            if file_num >= 25000:
                file_num = -1
                break
            """
    except Exception as e:
        print(e)
        print(f"{file_path} failed to load")

    return file_num

#split_audio_overlap('gtzan/data/genres/blues/blues.00000.wav', 100, 50, file_num)

In [40]:
#Fetching file names for original files
data_file = "Data/features_30_sec.csv"
music_data = pd.read_csv(data_file)
music_data

,filename,length,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,...,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,label
0,blues.00000.wav,661794,0.350088,0.088757,0.130228,0.002827,1784.165850,129774.064525,2002.449060,85882.761315,...,52.420910,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
1,blues.00001.wav,661794,0.340914,0.094980,0.095948,0.002373,1530.176679,375850.073649,2039.036516,213843.755497,...,55.356403,-0.731125,60.314529,0.295073,48.120598,-0.283518,51.106190,0.531217,45.786282,blues
2,blues.00002.wav,661794,0.363637,0.085275,0.175570,0.002746,1552.811865,156467.643368,1747.702312,76254.192257,...,40.598766,-7.729093,47.639427,-1.816407,52.382141,-3.439720,46.639660,-2.231258,30.573025,blues
3,blues.00003.wav,661794,0.404785,0.093999,0.141093,0.006346,1070.106615,184355.942417,1596.412872,166441.494769,...,44.427753,-3.319597,50.206673,0.636965,37.319130,-0.619121,37.259739,-3.407448,31.949339,blues
4,blues.00004.wav,661794,0.308526,0.087841,0.091529,0.002303,1835.004266,343399.939274,1748.172116,88445.209036,...,86.099236,-5.454034,75.269707,-0.916874,53.613918,-4.404827,62.910812,-11.703234,55.195160,blues
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,rock.00095.wav,661794,0.352063,0.080487,0.079486,0.000345,2008.149458,282174.689224,2106.541053,88609.749506,...,45.050526,-13.289984,41.754955,2.484145,36.778877,-6.713265,54.866825,-1.193787,49.950665,rock
996,rock.00096.wav,661794,0.398687,0.075086,0.076458,0.000588,2006.843354,182114.709510,2068.942009,82426.016726,...,33.851742,-10.848309,39.395096,1.881229,32.010040,-7.461491,39.196327,-2.795338,31.773624,rock
997,rock.00097.wav,661794,0.432142,0.075268,0.081651,0.000322,2077.526598,231657.968040,1927.293153,74717.124394,...,33.597008,-12.845291,36.367264,3.440978,36.001110,-12.588070,42.502201,-2.106337,29.865515,rock
998,rock.00098.wav,661794,0.362485,0.091506,0.083860,0.001211,1398.699344,240318.731073,1818.450280,109090.207161,...,46.324894,-4.416050,43.583942,1.556207,34.331261,-5.041897,47.227180,-3.590644,41.299088,rock


In [41]:
#Grouping up the different file names by genre
blues = music_data[music_data.label == 'blues']['filename'].tolist()
blues_files = [f"gtzan/data/genres/blues/{file}" for file in blues]

classical = music_data[music_data.label == 'classical']['filename'].tolist()
classical_files = [f"gtzan/data/genres/classical/{file}" for file in classical]

country = music_data[music_data.label == 'country']['filename'].tolist()
country_files = [f"gtzan/data/genres/country/{file}" for file in country]

disco = music_data[music_data.label == 'disco']['filename'].tolist()
disco_files = [f"gtzan/data/genres/disco/{file}" for file in disco]

hiphop = music_data[music_data.label == 'hiphop']['filename'].tolist()
hiphop_files = [f"gtzan/data/genres/hiphop/{file}" for file in hiphop]

jazz = music_data[music_data.label == 'jazz']['filename'].tolist()
jazz_files = [f"gtzan/data/genres/jazz/{file}" for file in jazz]

metal = music_data[music_data.label == 'metal']['filename'].tolist()
metal_files = [f"gtzan/data/genres/metal/{file}" for file in metal]

pop = music_data[music_data.label == 'pop']['filename'].tolist()
pop_files = [f"gtzan/data/genres/pop/{file}" for file in pop]

reggae = music_data[music_data.label == 'reggae']['filename'].tolist()
reggae_files = [f"gtzan/data/genres/reggae/{file}" for file in reggae]

rock = music_data[music_data.label == 'rock']['filename'].tolist()
rock_files = [f"gtzan/data/genres/rock/{file}" for file in rock]

In [42]:
# Create a generator with a specific seed
rng = np.random.default_rng(seed=42)

blues_rng = np.array(blues_files)
rng.shuffle(blues_rng)

classical_rng = np.array(classical_files)
rng.shuffle(classical_rng)

country_rng = np.array(country_files)
rng.shuffle(country_rng)

disco_rng = np.array(disco_files)
rng.shuffle(disco_rng)

hiphop_rng = np.array(hiphop_files)
rng.shuffle(hiphop_rng)

jazz_rng = np.array(jazz_files)
rng.shuffle(jazz_rng)

metal_rng = np.array(metal_files)
rng.shuffle(metal_rng)

pop_rng = np.array(pop_files)
rng.shuffle(pop_rng)

reggae_rng = np.array(reggae_files)
rng.shuffle(reggae_rng)

rock_rng = np.array(rock_files)
rng.shuffle(rock_rng)

blues_rng, rock_rng

(array(['gtzan/data/genres/blues/blues.00059.wav',
        'gtzan/data/genres/blues/blues.00021.wav',
        'gtzan/data/genres/blues/blues.00056.wav',
        'gtzan/data/genres/blues/blues.00018.wav',
        'gtzan/data/genres/blues/blues.00033.wav',
        'gtzan/data/genres/blues/blues.00042.wav',
        'gtzan/data/genres/blues/blues.00050.wav',
        'gtzan/data/genres/blues/blues.00027.wav',
        'gtzan/data/genres/blues/blues.00092.wav',
        'gtzan/data/genres/blues/blues.00071.wav',
        'gtzan/data/genres/blues/blues.00002.wav',
        'gtzan/data/genres/blues/blues.00025.wav',
        'gtzan/data/genres/blues/blues.00024.wav',
        'gtzan/data/genres/blues/blues.00052.wav',
        'gtzan/data/genres/blues/blues.00081.wav',
        'gtzan/data/genres/blues/blues.00099.wav',
        'gtzan/data/genres/blues/blues.00051.wav',
        'gtzan/data/genres/blues/blues.00061.wav',
        'gtzan/data/genres/blues/blues.00004.wav',
        'gtzan/data/genres/blue

In [43]:
# Example: 10s segments with 2s overlap (10s = 10000)
# Going for 30 ms segments with 10 ms overlap
# input format: "gtzan/data/genres/blues/blues.00000.wav"
#In ms
#300 = 0.3 seconds
#100 = 0.1 seconds
segment_length = 3000
#In ms
overlap_length = 1500

file_num = 0
#Blues
print("Starting Blues")
for file in blues_rng:
    curr_num = split_audio_overlap(file, segment_length, overlap_length, file_num)
    if(curr_num == -1):
        break
    file_num = curr_num
print("Finished Blues")

file_num = 0
#Classical
print("Starting Classical")
for file in classical_rng:
    curr_num = split_audio_overlap(file, segment_length, overlap_length, file_num)
    if(curr_num == -1):
        break
    file_num = curr_num
print("Finished Classical")
file_num = 0
#Country
print("Starting Country")
for file in country_rng:
    curr_num = split_audio_overlap(file, segment_length, overlap_length, file_num)
    if(curr_num == -1):
        break
    file_num = curr_num
print("Finished Country")
file_num = 0
#Disco
print("Starting Disco")
for file in disco_rng:
    curr_num = split_audio_overlap(file, segment_length, overlap_length, file_num)
    if(curr_num == -1):
        break
    file_num = curr_num
print("Finished Disco")
file_num = 0
#Hiphop
print("Starting Hiphop")
for file in hiphop_rng:
    curr_num = split_audio_overlap(file, segment_length, overlap_length, file_num)
    if(curr_num == -1):
        break
    file_num = curr_num
print("Finished Hiphop")
file_num = 0
#Jazz
print("Starting Jazz")
for file in jazz_rng:
    curr_num = split_audio_overlap(file, segment_length, overlap_length, file_num)
    if(curr_num == -1):
        break
    file_num = curr_num
print("Finished Jazz")
file_num = 0
#Metal
print("Starting Metal")
for file in metal_rng:
    curr_num = split_audio_overlap(file, segment_length, overlap_length, file_num)
    if(curr_num == -1):
        break
    file_num = curr_num
print("Finished Metal")
file_num = 0
#Pop
print("Starting Pop")
for file in pop_rng:
    curr_num = split_audio_overlap(file, segment_length, overlap_length, file_num)
    if(curr_num == -1):
        break
    file_num = curr_num
print("Finished Pop")
file_num = 0
#Reggae
print("Starting Reggae")
for file in reggae_rng:
    curr_num = split_audio_overlap(file, segment_length, overlap_length, file_num)
    if(curr_num == -1):
        break
    file_num = curr_num
print("Finished Reggae")
file_num = 0
#Rock
print("Starting Rock")
for file in rock_rng:
    curr_num = split_audio_overlap(file, segment_length, overlap_length, file_num)
    if(curr_num == -1):
        break
    file_num = curr_num
print("Finished Rock")


Starting Blues
Finished Blues
Starting Classical
Finished Classical
Starting Country
Finished Country
Starting Disco
Finished Disco
Starting Hiphop
Finished Hiphop
Starting Jazz
[Errno 2] No such file or directory: np.str_('gtzan/data/genres/jazz/jazz.00054.wav')
gtzan/data/genres/jazz/jazz.00054.wav failed to load
Finished Jazz
Starting Metal
Finished Metal
Starting Pop
Finished Pop
Starting Reggae
Finished Reggae
Starting Rock
Finished Rock
